In [7]:
!pip install -q ultralytics mediapipe opencv-python

In [6]:
import cv2
import mediapipe as mp  # 스켈레톤을 추출하는 라이브러리
import numpy as np
import csv  # csv 저장을 위해 라이브러리 추가
import os   # 파일 경로 관리를 위해 라이브러리 추가

# MediaPipe Pose 모델 초기화
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(
    min_detection_confidence = 0.7,  # 감지 최소 신뢰도
    min_tracking_confidence = 0.3 # 추적 최소 신뢰도
)

# MediaPipe 그리기 유틸리티 초기화
mp_drawing = mp.solutions.drawing_utils

# 동영상 파일 경로
video_path = "aespa_test.mp4" # 분석할 동영상 파일 경로 입력
cap = cv2.VideoCapture(video_path)

# 출력 파일 이름 설정
output_filename = os.path.splitext(os.path.basename(video_path))[0] + "_skeleton.csv"
# CSV 파일 헤더 준비 (33개 랜드마크 * 4개 좌표)
landmarks = ['class'] + [f'{j}_{i}' for i in mp_pose.PoseLandmark._member_names_ for j in ('x', 'y', 'z', 'v')]

# 동영상 파일이 정상적으로 열렸는지 확인
if not cap.isOpened():
    print(f"오류: '{video_path}' 동영상을 열 수 없습니다.")
    exit()

print("스켈레톤 추출을 시작합니다. 종료하려면 'q' 키를 누르세요.")

with open(output_filename, 'w', newline='') as f:
    csv_writer = csv.writer(f)
    csv_writer.writerow(landmarks)  # 헤더 작성

    print(f"'{output_filename}' 파일에 스켈레톤 데이터 저장을 시작합니다.")
    frame_count = 0
    while cap.isOpened():
        # 동영상에서 프레임 읽기
        success, image = cap.read()

        if not success:
            print("동영상 스트림의 끝에 도달했거나 오류가 발생했습니다.")
            break

        # 성능 향상을 위해 이미지를 읽기 전용으로 표시
        image.flags.writeable = False
        # BGR 이미지를 RGB로 변환
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # MediaPipe Pose를 사용하여 포즈 감지 수행
        results = pose.process(image_rgb)

        # 이미지를 다시 쓰기 가능으로 변경
        image.flags.writeable = True

        # 감지된 스켈레톤(포즈 랜드마크)을 원본 이미지에 그리기
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(
                image,
                results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS,
                landmark_drawing_spec = mp_drawing.DrawingSpec(color = (245, 117, 66), thickness=2, circle_radius=2),
                connection_drawing_spec=mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=2)
            )

            # 랜드마크 데이터 추출 및 csv 행으로 변환
            try:
                # 'dance' 클래스로 분류 (필요에 따라 변경 가능)
                class_name = "dance"

                # 모든 랜드마크의 x, y, z, v 값을 순서대로 리스트에 담기
                pose_row = list(np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten())

                # 클래스 이름과 랜드마크 데이터를 합쳐서 한 행으로 만듦
                row = [class_name] + pose_row
                
                # csv 파일에 한 행 쓰기
                csv_writer.writerow(row)

            except Exception as e:
                print(f"프레임 {frame_count} 처리 중 오류 발생: {e}")
                pass # 오류 발생 시 해당 프레임 건너뜀

        # 결과 영상 출력
        cv2.imshow('MediaPipe Pose Skeleton', image)

        frame_count += 1
        # 'q' 키를 누르면 루프 종료
        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

# 자원 해제
cap.release()
cv2.destroyAllWindows()
pose.close()

print("스켈레톤 추출이 완료되었습니다.")

스켈레톤 추출을 시작합니다. 종료하려면 'q' 키를 누르세요.
'aespa_test_skeleton.csv' 파일에 스켈레톤 데이터 저장을 시작합니다.
스켈레톤 추출이 완료되었습니다.


In [ ]:
import cs2
import mediapipe as mp
from ultralytics import YOLO
import numpy as np

# ----------------
# 1. 초기 설정
# ----------------

# YOLOv8 모델 로드 (가장 작고 빠른 'n' 모델 사용)
yolo_model = YOLO('yolov8n.pt')

# MediaPipe Pose 모델 초기화
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)
mp_drawing = mp.solutions.drawing_utils

# 동영상 파일 열기
video_path = "TWICE_ICANTSTOPME.mp4"    # 스켈레톤 추출할 동영상 파일
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print(f"오류: '{video_path}' 동영상을 열 수 없습니다.")
    exit()

# 결과 동영상을 저장하기 위한 설정
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
output_path = video_path.replace(".mp4", "_output.mp4")
out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))

print("Top-down 방식 스켈레톤 추출을 시작합니다. 결과는 동영상 파일로 저장됩니다.")

# --------------------------
# 2. 메인 루프: 프레임별 처리
# --------------------------
while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    # STEP 1: 사람 탐지 (YOLOv8)
    # 현재 프레임에서 모든 객체를 탐지
    yolo_results = yolo_model(frame, classes=[0], conf=0.6, verbose=False)

    # 탐지된 각 사람에 대해 반복 처리
    for result in yolo_results:
        for box in result.boxes:
            # 바운딩 박스 좌표 추출 (x1, y1, x2, y2)
            cords = box.xyxy[0].tolist()
            cords = [int(x) for x in cords]
            x1, y1, x2, y2 = cords

            # STEP 2: 개별 영역 추출 및 포즈 추정 (Crop & MediaPipe)
            # 탐지된 사람의 영역만 잘라내기
            person_crop = frame[y1:y2, x1:x2]

            # 잘라낸 이미지가 비어있지 않은지 확인
            if person_crop.shape[0] == 0 or person_crop.shape[1] == 0:
                continue

            # MediaPipe Pose는 RGB 이미지를 입력으로 받음
            crop_rgb = cv2.cvtColor(person_crop, cv2.COLOR_BGR2RGB)
            pose_results = pose.process(crop_rgb)

            # STEP 3: 좌표 변환 및 시각화
            if pose_results.pose_landmarks:
                # 원본 프레임에 YOLO 바운딩 박스 그리기
                cv2.rectangle(frame, (x1,y1), (x2, y2), (0, 255, 0), 2)

                # MediaPipe 랜드마크를 원본 프레임 좌표로 변환
                landmarks = pose_results.pose_landmarks
                crop_h, crop_w, _ = person_crop.shape

                # 변환된 랜드마크를 그리기 위해 새로운 랜드마크 객체 생성
                # landmarks.landmark는 직접 수정이 어려워 복사본을 만들어 처리
                world_landmarks = []
                for landmark in landmarks.landmark:
                    world_landmark = {
                        'x': x1 + landmark.x * crop_w,
                        'y': y1 + landmark.y * crop_h,
                        'z': landmark.z,    # z는 변환 없이 사용 가능
                        'visibility': landmark.visibility
                    }
                    world_landmarks.append(world_landmark)

                    # 각 점과 선을 직접 그려 더 명확하게 제어
                    # 각 랜드마크(점) 그리기
                    for landmark in world_landmarks:
                        if landmark['visibility'] > 0.5: # 잘 보이는 점만 그리기
                            cv2.circle(frame, (int(landmark['x']), int(landmark['y'])), 3, (255, 0, 0), -1)


